<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "566"
iso_country = "Nigeria"
year = 2024
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 243.300*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Nigeria': 

Number of trade records present in dataset: 24908
Number of partner countries present in dataset: 156
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$36812.2m
Total weight within the dataset: 42684.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,566,Nigeria,156,China,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,3.099181e+05,NaN,3.099181e+05,kg,174757.0,174757.0,5.125735e+05,20197.0,12549.830087,6.432710e+06,['Container'],['Container'],20144.397967,20051.512721,20057.648179,20057.648179,20121.062705,20121.062705,21196.621121,4.089540e+04,23694.893534,21440.914664,21533.763441,22259.835076,20697.664775,3.847199e+04,24407.309394,22380.358870,21890.864916,23024.452795,20260.505666,4.250477e+04,30675.572760,26620.002576,24620.537781,24081.668115,19828.568911,4.516289e+04,3.575950e+04,3.361138e+04,26144.501017,25936.254270,19404.571500,4.616319e+04,4.278051e+04,4.349682e+04,28414.617697,27297.107134,18972.650341,4.529323e+04,4.415444e+04,4.626444e+04,3.411642e+04,27815.547432,18560.326350,4.034368e+04,4.081152e+04,4.311716e+04,32793.420337,28161.371267,18148.146127,3.580401e+04,3.696434e+04,4.302547e+04,3.492830e+04,28391.040364,2322
1,1,566,Nigeria,792,TÃ¼rkiye,Import,Sea,110812,110812 | Starch; maize (corn),1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,2.098378e+07,NaN,2.098378e+07,kg,43319129.0,43319129.0,3.470508e+07,9015.0,5601.659565,1.944060e+08,['Container'],['Container'],608793.575403,605986.445568,606171.868183,606171.868183,608088.349203,608088.349203,640593.319292,1.235920e+06,716094.815888,647976.232378,650782.260089,672725.221497,625514.118688,1.162681e+06,737625.079519,676367.629261,661574.396157,695833.102688,612302.522225,1.284557e+06,927061.292634,804495.948332,744069.157542,727783.717234,599248.753056,1.364889e+06,1.080705e+06,1.015786e+06,790125.586170,783832.060698,586434.922599,1.395120e+06,1.292891e+06,1.314538e+06,858731.878230,824959.052032,573381.624756,1.368829e+06,1.334413e+06,1.398180e+06,1.031049e+06,840627.086538,560920.582373,1.219246e+06,1.233385e+06,1.303065e+06,991065.786624,851078.395613,548463.884885,1.082050e+06,1.117117e+06,1.300294e+06,1.055585e+06,858019.336265,2322


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $4552.3m (12.4%), ... 

                               cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                               
10 | Cereals               1.211599e+09  4.397157e+09      3.3     10.3
17 | Sugars and sugar con  6.975155e+08  1.449256e+09      1.9      3.4
03 | Fish and crustaceans  6.818353e+08  2.913358e+08      1.9      0.7
19 | Preparations of cere  3.967974e+08  1.238702e+08      1.1      0.3
04 | Dairy produce; birds  3.174401e+08  1.273535e+08      0.9      0.3
21 | Miscellaneous edible  2.479250e+08  6.189165e+07      0.7      0.1
15 | Animal, vegetable or  2.322771e+08  2.362054e+08      0.6      0.6
11 | Products of the mill  2.292449e+08  3.789980e+08      0.6      0.9
22 | Beverages, spirits a  2.008549e+08  1.750423e+08      0.5      0.4
20 | Preparations of vege  9.280668e+07  7.262211e+07      0.3      0.2
23 | Food industries, res  7.37757

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $2337.2m (6.3%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Food & Beverages  1.135233e+09  5.881643e+08      3.1      1.4
Agriculture       1.081007e+09  1.426512e+09      2.9      3.3
Fishing           1.210065e+08  5.774362e+07      0.3      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $2108.0m (5.7%), ... 

                                   cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                 
OFD - Food products n.e.c.     1.114743e+09  587874412.0      3.0      1.4
MIL - Dairy products           3.252810e+08  140415419.9      0.9      0.3
VOL - Vegetable oils and fats  2.322718e+08  242305576.1      0.6      0.6
GRO - Cereal grains n.e.c.     1.293317e+08  534673193.8      0.4      1.3
OCR - Crops n.e.c.             9.199570e+07   28507793.8      0.2      0.1
OAP - Animal products n.e.c.   7.230513e+07   50177922.2      0.2      0.1
V_F - Vegetables, fruit, nuts  6.560258e+07   60959839.7      0.2      0.1
PDR - Paddy rice               2.803985e+07   48705212.8      0.1      0.1
OMT - Meat products n.e.c.     2.302300e+07   19452793.3      0.1      0.0
PFB - Plant-based fibers       1.641828e+07   21932001.2      0.0      0.1
PCR - Processed rice           6.3

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 12.4% 
GTAP 5.7% 
EORA 6.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46407/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,566,Nigeria,156,China,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,3.099181e+05,NaN,3.099181e+05,kg,174757.0,174757.0,5.125735e+05,20197.0,12549.830087,6.432710e+06,['Container'],['Container'],20144.397967,20051.512721,20057.648179,20057.648179,20121.062705,20121.062705,21196.621121,4.089540e+04,23694.893534,21440.914664,21533.763441,22259.835076,20697.664775,3.847199e+04,24407.309394,22380.358870,21890.864916,23024.452795,20260.505666,4.250477e+04,30675.572760,26620.002576,24620.537781,24081.668115,19828.568911,4.516289e+04,3.575950e+04,3.361138e+04,26144.501017,25936.254270,19404.571500,4.616319e+04,4.278051e+04,4.349682e+04,28414.617697,27297.107134,18972.650341,4.529323e+04,4.415444e+04,4.626444e+04,3.411642e+04,27815.547432,18560.326350,4.034368e+04,4.081152e+04,4.311716e+04,32793.420337,28161.371267,18148.146127,3.580401e+04,3.696434e+04,4.302547e+04,3.492830e+04,28391.040364,2322,Processed Cereals
1,1,566,Nigeria,792,TÃ¼rkiye,Import,Sea,110812,110812 | Starch; maize (corn),1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,2.098378e+07,NaN,2.098378e+07,kg,43319129.0,43319129.0,3.470508e+07,9015.0,5601.659565,1.944060e+08,['Container'],['Container'],608793.575403,605986.445568,606171.868183,606171.868183,608088.349203,608088.349203,640593.319292,1.235920e+06,716094.815888,647976.232378,650782.260089,672725.221497,625514.118688,1.162681e+06,737625.079519,676367.629261,661574.396157,695833.102688,612302.522225,1.284557e+06,927061.292634,804495.948332,744069.157542,727783.717234,599248.753056,1.364889e+06,1.080705e+06,1.015786e+06,790125.586170,783832.060698,586434.922599,1.395120e+06,1.292891e+06,1.314538e+06,858731.878230,824959.052032,573381.624756,1.368829e+06,1.334413e+06,1.398180e+06,1.031049e+06,840627.086538,560920.582373,1.219246e+06,1.233385e+06,1.303065e+06,991065.786624,851078.395613,548463.884885,1.082050e+06,1.117117e+06,1.300294e+06,1.055585e+06,858019.336265,2322,Processed Cereals


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $3273.5m of $36812.2m (8.9%)
Volume: 6714.0 of 42684.7 million kilograms (15.7%)
Value-to-Weight: 0.49 from 0.86 US$ per kg (+-43.5%)
Average distance to market (volume-weighted): 5631.6 from 6308.6 nm (+-10.7%)

            food_group  N_com      cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     42  1.177214e+09  4.337430e+09      3.2     10.2
1    Processed Cereals    432  8.457868e+08  6.006977e+08      2.3      1.4
2      Sugar and Honey     89  6.977309e+08  1.449318e+09      1.9      3.4
3                Dairy    147  3.122373e+08  1.257509e+08      0.8      0.3
4        Fats and Oils     73  7.991943e+07  5.694399e+07      0.2      0.1
5           Vegetables    159  7.917547e+07  7.065258e+07      0.2      0.2
6               Fruits    118  5.385062e+07  4.443382e+07      0.1      0.1
7        Meat and Prep     30  2.761464e+07  2.878560e+07      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,3.313504e+08,6.382257e+08,3.701630e+08,3.351444e+08,3.365883e+08,3.479095e+08,3.034350e+08,7.206798e+08,6.679710e+08,6.792846e+08,4.437312e+08,4.264948e+08,2.838363e+08,5.591792e+08,5.772318e+08,6.717877e+08,5.454019e+08,4.436383e+08
1,Fats and Oils,1.308557e+08,2.269127e+08,1.417423e+08,1.317825e+08,1.323439e+08,1.364667e+08,1.216984e+08,2.656104e+08,2.469636e+08,2.567515e+08,1.702833e+08,1.635333e+08,1.151831e+08,2.080512e+08,2.127773e+08,2.471168e+08,2.018701e+08,1.689550e+08
2,Fruits,8.563096e+07,1.620115e+08,9.513805e+07,8.658646e+07,8.693760e+07,8.979287e+07,7.864654e+07,1.832781e+08,1.702133e+08,1.733596e+08,1.138884e+08,1.095666e+08,7.373270e+07,1.433429e+08,1.476729e+08,1.715376e+08,1.392355e+08,1.139158e+08
3,Meat and Prep,4.915615e+07,9.483872e+07,5.494978e+07,4.972268e+07,4.993800e+07,5.162180e+07,4.500029e+07,1.070550e+08,9.921040e+07,1.008715e+08,6.589509e+07,6.330352e+07,4.208657e+07,8.303150e+07,8.572236e+07,9.977848e+07,8.100067e+07,6.584041e+07
4,Processed Cereals,1.336205e+09,2.527100e+09,1.483536e+09,1.350027e+09,1.355674e+09,1.400765e+09,1.227499e+09,2.868972e+09,2.662578e+09,2.714006e+09,1.780752e+09,1.708684e+09,1.150826e+09,2.234904e+09,2.303520e+09,2.676309e+09,2.177001e+09,1.776542e+09
5,Sugar and Honey,1.743590e+09,2.769128e+09,1.840440e+09,1.737223e+09,1.742739e+09,1.803604e+09,1.644692e+09,3.305920e+09,3.150899e+09,3.255088e+09,2.310180e+09,2.079541e+09,1.575554e+09,2.703388e+09,2.744442e+09,3.121516e+09,2.640441e+09,2.133874e+09
6,Unprocessed Cereals,8.326140e+09,1.573897e+10,9.248156e+09,8.407584e+09,8.442889e+09,8.728774e+09,7.648724e+09,1.786422e+10,1.660329e+10,1.690907e+10,1.114697e+10,1.063162e+10,7.173913e+09,1.393201e+10,1.435808e+10,1.667240e+10,1.359324e+10,1.104178e+10
7,Vegetables,1.521995e+08,2.865113e+08,1.686671e+08,1.537900e+08,1.544254e+08,1.595019e+08,1.399359e+08,3.257976e+08,3.024061e+08,3.084573e+08,2.021679e+08,1.944826e+08,1.312146e+08,2.538216e+08,2.616020e+08,3.040127e+08,2.471261e+08,2.022354e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.98,0.12,0.01,0.02,0.05,1.34,1.17,1.20,0.45,0.39,0.88,0.94,1.24,0.84,0.51
1,Fats and Oils,1.20,0.14,0.01,0.02,0.07,1.80,1.57,1.69,0.61,0.52,1.16,1.22,1.65,1.08,0.67
2,Fruits,1.42,0.18,0.02,0.02,0.08,1.94,1.70,1.76,0.65,0.57,1.29,1.37,1.82,1.22,0.75
3,Meat and Prep,1.65,0.21,0.02,0.03,0.09,2.25,1.96,2.02,0.76,0.66,1.48,1.58,2.09,1.41,0.86
4,Processed Cereals,1.41,0.17,0.02,0.02,0.08,1.94,1.70,1.76,0.65,0.57,1.28,1.36,1.80,1.21,0.74
5,Sugar and Honey,1.47,0.14,-0.01,-0.00,0.09,2.38,2.16,2.31,0.95,0.62,1.62,1.68,2.22,1.53,0.80
6,Unprocessed Cereals,6.30,0.78,0.07,0.10,0.34,8.68,7.61,7.87,2.97,2.53,5.74,6.10,8.07,5.45,3.29
7,Vegetables,1.70,0.21,0.02,0.03,0.09,2.35,2.05,2.13,0.79,0.69,1.55,1.65,2.18,1.46,0.90
